# Distribution check for `benchmark.html`

Recomputes the per-direction percentages and per-treatment counts shown on the
onboarding `benchmark.html` page so the figures can be verified end-to-end.

Run from `ai-research/experiments_05292026/` (where this notebook lives).

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Pull the canonical taxonomy + direction mapping from the pipeline so this
# notebook tracks the source of truth.
REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'citator-pipeline'))
from utils.postprocess import TREATMENT_RANK, direction_mapping, severity_mapping

LABELS = REPO_ROOT / 'data' / 'benchmark_original' / '0410.csv'
df = pd.read_csv(LABELS)
print(f'Total rows in 0410.csv: {len(df):,}')
df.head()

Total rows in 0410.csv: 14,536


,citing-cited,citing_cluster_id,cited_cluster_id,batch,cited_case_name,cited_citation_strings,expert_treatment,revised_treatment,change_treatment,final_treatment,severity,direction,examples
0,100397-100087,100397,100087.0,SCOTUS,Galveston Wharf Co. v. City of Galveston,"260 U.S. 473, 43 S. Ct. 168, 67 L. Ed. 355, 19...",Cited by,Cited by,False,Cited by,Neutral,Citing Reference,False
1,100397-100149,100397,100149.0,SCOTUS,Seaboard Air Line Railway Co. v. United States,"261 U.S. 299, 43 S. Ct. 354, 67 L. Ed. 664, 19...",Cited by,Cited by,False,Cited by,Neutral,Citing Reference,False
2,100397-100256,100397,100256.0,SCOTUS,Rindge Co. v. County of Los Angeles,"262 U.S. 700, 43 S. Ct. 689, 67 L. Ed. 1186, 1...",Cited by,Cited by,False,Cited by,Neutral,Citing Reference,False
3,100397-3643566,100397,3643566.0,SCOTUS,Burbank v. . Fay,65 N.Y. 57,Cited by,Cited by,False,Cited by,Neutral,Citing Reference,False
4,100397-85452,100397,85452.0,SCOTUS,US Bank v. PLANTERS'BANK,"6 L. Ed. 244, 1824 U.S. LEXIS 410, 22 U.S. 904...",Cited by,Cited by,False,Cited by,Neutral,Citing Reference,False


## Filter to usable rows

Keep only rows whose `final_treatment` is one of the canonical treatment labels
(22 base treatments + 21 "as recognized by" variants = 43 total). Rows marked
`MANUAL` / `PENDING` / `MISSING` / blank are dropped.

In [2]:
# Build the full 43-label set: 22 base treatments + 21 RR variants (all except 'Cited by').
BASE = list(TREATMENT_RANK.keys())
RR_VARIANTS = [
    f'{t.removesuffix(" by")} as recognized by'
    for t in BASE
    if t != 'Cited by'
]
CANONICAL = set(BASE) | set(RR_VARIANTS)
assert len(CANONICAL) == 43, f'expected 43 canonical labels, got {len(CANONICAL)}'

usable = df[df['final_treatment'].isin(CANONICAL)].copy()
print(f'Usable labeled records: {len(usable):,}')

Usable labeled records: 9,740


## Direction lookup

`direction_mapping` covers the 22 base treatments (DH vs CR). Any treatment
ending in `as recognized by` is Related Reference by definition.

In [3]:
def lookup_direction(t: str) -> str:
    if t.endswith('as recognized by'):
        return 'Related Reference'
    return direction_mapping[t]

def lookup_severity(t: str) -> str:
    if t.endswith('as recognized by'):
        base = t.removesuffix(' as recognized by') + ' by'
        return severity_mapping[base]
    return severity_mapping[t]

usable['direction'] = usable['final_treatment'].map(lookup_direction)
usable['severity'] = usable['final_treatment'].map(lookup_severity)

## Per-direction summary

Matches the **direction bar** on `benchmark.html`.

In [4]:
totals = usable['direction'].value_counts()
pct = (totals / totals.sum() * 100).round(2)
summary = pd.DataFrame({'count': totals, 'pct': pct})
summary.loc['TOTAL'] = [totals.sum(), 100.0]
summary

,count,pct
direction,,
Citing Reference,9397.0,96.48
Related Reference,220.0,2.26
Direct History,123.0,1.26
TOTAL,9740.0,100.00


## Per-treatment counts (with severity + direction)

Matches the **3-column distribution graphs** on `benchmark.html`. All 43
canonical treatments are included; ones not present in the benchmark show
count = 0.

Sorted within each direction by severity tier (Stop → Warning → Caution →
Neutral), then by the canonical `TREATMENT_RANK` order.

In [5]:
TIER_ORDER = {'Stop': 0, 'Warning': 1, 'Caution': 2, 'Neutral': 3}

def rank(t: str) -> int:
    if t.endswith('as recognized by'):
        base = t.removesuffix(' as recognized by') + ' by'
        return TREATMENT_RANK[base]
    return TREATMENT_RANK[t]

counts = usable['final_treatment'].value_counts()

rows = []
for t in sorted(CANONICAL, key=lambda x: (lookup_direction(x), TIER_ORDER[lookup_severity(x)], rank(x))):
    rows.append({
        'direction': lookup_direction(t),
        'severity': lookup_severity(t),
        'treatment': t,
        'count': int(counts.get(t, 0)),
    })

per_treatment = pd.DataFrame(rows)
per_treatment['pct_of_total'] = (per_treatment['count'] / per_treatment['count'].sum() * 100).round(3)
per_treatment

,direction,severity,treatment,count,pct_of_total
0,Citing Reference,Stop,Overruled by,11,0.113
1,Citing Reference,Stop,Abrogated by,11,0.113
2,Citing Reference,Stop,Questioned by,2,0.021
3,Citing Reference,Warning,Disapproved by,2,0.021
4,Citing Reference,Warning,Limited by,4,0.041
5,Citing Reference,Caution,Criticized by,4,0.041
6,Citing Reference,Caution,Distinguished by,196,2.012
7,Citing Reference,Caution,Declined to follow by,3,0.031
8,Citing Reference,Neutral,Cited by,9164,94.086
9,Direct History,Stop,Reversed by,36,0.370


## Bar-width formula

The HTML uses log-scaled widths: `width% = log(n+1) / log(max+1) * 100`,
rounded. `max` = 9,164 (the `Cited by` count). Zero-count treatments get
width 0%.

In [6]:
import math
MAX_N = per_treatment['count'].max()

def bar_width(n: int) -> int:
    if n == 0:
        return 0
    return round(math.log(n + 1) / math.log(MAX_N + 1) * 100)

per_treatment['bar_width_pct'] = per_treatment['count'].map(bar_width)
per_treatment

,direction,severity,treatment,count,pct_of_total,bar_width_pct
0,Citing Reference,Stop,Overruled by,11,0.113,27
1,Citing Reference,Stop,Abrogated by,11,0.113,27
2,Citing Reference,Stop,Questioned by,2,0.021,12
3,Citing Reference,Warning,Disapproved by,2,0.021,12
4,Citing Reference,Warning,Limited by,4,0.041,18
5,Citing Reference,Caution,Criticized by,4,0.041,18
6,Citing Reference,Caution,Distinguished by,196,2.012,58
7,Citing Reference,Caution,Declined to follow by,3,0.031,15
8,Citing Reference,Neutral,Cited by,9164,94.086,100
9,Direct History,Stop,Reversed by,36,0.370,40


## Scarce-tag check

Treatments flagged as **scarce** on `benchmark.html` are those with fewer
than 4 labeled rows (count ≤ 3) — too thin to produce a reliable per-class
metric. This matches the page's "<4 labeled examples" badge threshold.
Expect 24 of the 43 canonical treatments (13 with 0 records + 11 with 1–3)
to fall into this bucket.

In [7]:
scarce = per_treatment[per_treatment['count'] <= 3]
print(f'Scarce treatments (count 0–3): {len(scarce)}')
print(f'  of which count = 0: {(scarce["count"] == 0).sum()}')
print(f'  of which count 1–3: {scarce["count"].between(1, 3).sum()}')
scarce

Scarce treatments (count 0–3): 24
  of which count = 0: 13
  of which count 1–3: 11


,direction,severity,treatment,count,pct_of_total,bar_width_pct
2,Citing Reference,Stop,Questioned by,2,0.021,12
3,Citing Reference,Warning,Disapproved by,2,0.021,12
7,Citing Reference,Caution,Declined to follow by,3,0.031,15
12,Direct History,Stop,Vacated by,1,0.010,8
13,Direct History,Warning,Reversed in part; Vacated in part by,0,0.000,0
14,Direct History,Warning,Affirmed in part; Reversed in part by,3,0.031,15
15,Direct History,Warning,Affirmed in part; Vacated in part by,0,0.000,0
16,Direct History,Caution,Modified by,0,0.000,0
17,Direct History,Caution,Remanded by,0,0.000,0
18,Direct History,Caution,Cert. granted by,0,0.000,0
